# Baseline ML Pipeline — Emotion Recognition (DREAMER)
**Models:** Logistic Regression, SVM, Random Forest, GBM  
**Features:** 258 EEG + 22 ECG = 280 hand-crafted features  
**Targets:** Valence, Arousal, Dominance (binary classification)  
**Evaluation:** Within-subject + Cross-subject (LOSO)

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    GITHUB_REPO = "https://github.com/YOUR_USERNAME/emotion-recognition.git"
    !git clone {GITHUB_REPO} /content/emotion-recognition
    %cd /content/emotion-recognition
    !pip install mat73 pyyaml scikit-learn -q

    from google.colab import drive
    drive.mount("/content/drive")

    import shutil, os
    os.makedirs("data/raw", exist_ok=True)
    shutil.copy("/content/drive/MyDrive/DREAMER.mat", "data/raw/DREAMER.mat")
    print("✅ Ready")
else:
    %cd ..  # adjust if running locally
    print("✅ Running locally")

In [ ]:
import sys
sys.path.insert(0, ".")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json, os, time, warnings

warnings.filterwarnings("ignore")
sns.set_theme(style="darkgrid")
plt.rcParams.update({"figure.dpi": 120})

from src.utils.config      import load_config
from src.data.loader       import (
    load_dreamer_mat, get_subject_data,
    get_trial_signals, get_trial_labels,
    N_SUBJECTS, N_VIDEOS
)
from src.data.preprocessor import process_trial
from src.features.eeg_features import extract_eeg_features
from src.features.ecg_features import extract_ecg_features
from src.models.baseline   import build_baseline, evaluate, save_model

config = load_config("configs/default.yaml")
DREAMER_PATH = config["data"]["raw_path"]
THRESHOLD    = config["labels"]["threshold"]
TARGETS      = config["labels"]["targets"]
WINDOW_SEC   = config["data"]["segment_length"]
OVERLAP_SEC  = config["data"]["overlap"]
NORM_METHOD  = config["data"]["norm_method"]
EEG_FS       = config["data"]["sampling_rate_eeg"]
ECG_FS       = config["data"]["sampling_rate_ecg"]

print("✅ Config loaded")
print(f"   Targets     : {TARGETS}")
print(f"   Window      : {WINDOW_SEC}s | Overlap: {OVERLAP_SEC}s")
print(f"   Threshold   : {THRESHOLD}")

## 1. Build Full Feature Matrix
Extract 280 features per window across all subjects × videos × windows.
This cell takes ~5–10 minutes on Colab CPU.

In [ ]:
from tqdm.auto import tqdm

dreamer = load_dreamer_mat(DREAMER_PATH)

all_rows = []

for sub_idx in tqdm(range(N_SUBJECTS), desc="Subjects"):
    subject = get_subject_data(dreamer, sub_idx)

    for vid_idx in range(N_VIDEOS):
        try:
            eeg_s, ecg_s = get_trial_signals(subject, vid_idx, "stimuli")
            eeg_b, ecg_b = get_trial_signals(subject, vid_idx, "baseline")
            raw_labels   = get_trial_labels(subject, vid_idx)

            # Binary labels for all three targets
            bin_labels = {k: int(v > THRESHOLD)
                          for k, v in raw_labels.items()}

            # Preprocess + segment
            eeg_segs, ecg_segs = process_trial(
                eeg_s, ecg_s, eeg_b, ecg_b,
                eeg_fs=EEG_FS, ecg_fs=ECG_FS,
                window_sec=WINDOW_SEC,
                overlap_sec=OVERLAP_SEC,
                norm_method=NORM_METHOD,
            )

            for w in range(eeg_segs.shape[0]):
                eeg_feat = extract_eeg_features(eeg_segs[w], fs=EEG_FS)
                ecg_feat = extract_ecg_features(ecg_segs[w], fs=ECG_FS)
                feat_vec = np.concatenate([eeg_feat, ecg_feat])

                all_rows.append({
                    "subject"  : sub_idx + 1,
                    "video"    : vid_idx + 1,
                    "window"   : w,
                    "features" : feat_vec,
                    **bin_labels,
                })

        except Exception as e:
            print(f"  ⚠ Skipped sub={sub_idx+1} vid={vid_idx+1}: {e}")

print(f"\n✅ Total windows extracted: {len(all_rows)}")
print(f"   Feature vector size    : {all_rows[0]['features'].shape[0]}")

In [ ]:
subjects  = np.array([r["subject"]   for r in all_rows])
videos    = np.array([r["video"]     for r in all_rows])
X         = np.stack([r["features"]  for r in all_rows])

y = {
    "valence"  : np.array([r["valence"]   for r in all_rows]),
    "arousal"  : np.array([r["arousal"]   for r in all_rows]),
    "dominance": np.array([r["dominance"] for r in all_rows]),
}

print(f"X shape   : {X.shape}")
for t in TARGETS:
    pos = y[t].sum()
    print(f"y[{t}] — class 1: {pos} ({100*pos/len(y[t]):.1f}%)"
          f"  class 0: {len(y[t])-pos} ({100*(1-pos/len(y[t])):.1f}%)")

## 2. Within-Subject Evaluation
Train and test on each subject independently (80/20 split per subject).  
Gives the upper-bound performance — model has seen same subject's data.

In [ ]:
from sklearn.model_selection import train_test_split

MODEL_TYPES = ["logreg", "svm", "rf", "gbm"]
within_results = []

for target in TARGETS:
    print(f"\n{'='*55}")
    print(f"  Target: {target.upper()}")
    print(f"{'='*55}")

    for model_type in MODEL_TYPES:
        sub_accs, sub_f1s, sub_aucs = [], [], []

        for sub_id in range(1, N_SUBJECTS + 1):
            mask = subjects == sub_id
            X_sub = X[mask]
            y_sub = y[target][mask]

            if len(np.unique(y_sub)) < 2 or len(y_sub) < 10:
                continue   # skip subjects with single class

            X_tr, X_te, y_tr, y_te = train_test_split(
                X_sub, y_sub,
                test_size=0.2, random_state=42, stratify=y_sub
            )

            model = build_baseline(model_type)
            model.fit(X_tr, y_tr)
            m = evaluate(model, X_te, y_te, split=f"sub{sub_id}")

            sub_accs.append(m["accuracy"])
            sub_f1s.append(m["f1"])
            sub_aucs.append(m["roc_auc"])

        within_results.append({
            "target"  : target,
            "model"   : model_type,
            "eval"    : "within",
            "acc_mean": round(np.mean(sub_accs), 4),
            "acc_std" : round(np.std(sub_accs),  4),
            "f1_mean" : round(np.mean(sub_f1s),  4),
            "f1_std"  : round(np.std(sub_f1s),   4),
            "auc_mean": round(np.mean(sub_aucs), 4),
            "auc_std" : round(np.std(sub_aucs),  4),
        })

        print(f"  [{model_type.upper():6s}] "
              f"Acc={np.mean(sub_accs):.3f}±{np.std(sub_accs):.3f}  "
              f"F1={np.mean(sub_f1s):.3f}±{np.std(sub_f1s):.3f}  "
              f"AUC={np.mean(sub_aucs):.3f}±{np.std(sub_aucs):.3f}")

df_within = pd.DataFrame(within_results)
print("\n✅ Within-subject evaluation complete")

## 3. Cross-Subject Evaluation — Leave-One-Subject-Out (LOSO)
Train on 22 subjects, test on the held-out subject.  
Repeated for all 23 subjects → true generalisation performance.

In [ ]:
loso_results = []

for target in TARGETS:
    print(f"\n{'='*55}")
    print(f"  LOSO — Target: {target.upper()}")
    print(f"{'='*55}")

    for model_type in MODEL_TYPES:
        fold_accs, fold_f1s, fold_aucs = [], [], []

        for test_sub in range(1, N_SUBJECTS + 1):
            train_mask = subjects != test_sub
            test_mask  = subjects == test_sub

            X_tr, y_tr = X[train_mask], y[target][train_mask]
            X_te, y_te = X[test_mask],  y[target][test_mask]

            if len(np.unique(y_te)) < 2:
                continue

            model = build_baseline(model_type)
            model.fit(X_tr, y_tr)
            m = evaluate(model, X_te, y_te, split=f"loso_sub{test_sub}")

            fold_accs.append(m["accuracy"])
            fold_f1s.append(m["f1"])
            fold_aucs.append(m["roc_auc"])

        loso_results.append({
            "target"  : target,
            "model"   : model_type,
            "eval"    : "loso",
            "acc_mean": round(np.mean(fold_accs), 4),
            "acc_std" : round(np.std(fold_accs),  4),
            "f1_mean" : round(np.mean(fold_f1s),  4),
            "f1_std"  : round(np.std(fold_f1s),   4),
            "auc_mean": round(np.mean(fold_aucs), 4),
            "auc_std" : round(np.std(fold_aucs),  4),
        })

        print(f"  [{model_type.upper():6s}] "
              f"Acc={np.mean(fold_accs):.3f}±{np.std(fold_accs):.3f}  "
              f"F1={np.mean(fold_f1s):.3f}±{np.std(fold_f1s):.3f}  "
              f"AUC={np.mean(fold_aucs):.3f}±{np.std(fold_aucs):.3f}")

df_loso = pd.DataFrame(loso_results)
print("\n✅ LOSO evaluation complete")

## 4. Results Comparison — Within vs LOSO

In [ ]:
df_all = pd.concat([df_within, df_loso], ignore_index=True)

print("\n── Full Results Table ──")
print(df_all.to_string(index=False))

os.makedirs("outputs/results", exist_ok=True)
df_all.to_csv("outputs/results/baseline_results.csv", index=False)
print("\n✅ Saved: outputs/results/baseline_results.csv")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
metrics_to_plot = ["acc_mean", "f1_mean", "auc_mean"]
metric_labels   = ["Accuracy", "F1 Score", "ROC-AUC"]
colors_eval     = {"within": "#4C72B0", "loso": "#DD8452"}

for ax, metric, label in zip(axes, metrics_to_plot, metric_labels):
    for eval_type, grp in df_all.groupby("eval"):
        pivot = grp.pivot(index="model", columns="target", values=metric)
        x     = np.arange(len(pivot.columns))
        width = 0.18
        offsets = np.linspace(-width, width, len(pivot.index))

        for i, (mdl, row) in enumerate(pivot.iterrows()):
            ax.bar(x + offsets[i], row.values, width=width * 0.9,
                   label=f"{mdl}-{eval_type}" if metric == "acc_mean" else "",
                   alpha=0.85,
                   color=plt.cm.tab10(i / len(pivot.index)),
                   edgecolor="white",
                   linestyle="-" if eval_type == "within" else "--",
                   linewidth=0.5)

    ax.set_title(f"{label}", fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(TARGETS, fontweight="bold")
    ax.set_ylabel(label)
    ax.set_ylim(0, 1.05)
    ax.axhline(0.5, color="black", linestyle="--", linewidth=0.8,
               alpha=0.5, label="Chance")

axes[0].legend(fontsize=7, ncol=2, loc="upper right")
plt.suptitle("Baseline Model Comparison — Within vs LOSO",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/results/baseline_comparison.png", bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, eval_type in zip(axes, ["within", "loso"]):
    subset = df_all[df_all["eval"] == eval_type].copy()
    pivot  = subset.pivot(index="model", columns="target", values="acc_mean")
    sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlGn",
                vmin=0.4, vmax=1.0, ax=ax,
                linewidths=0.4, annot_kws={"size": 11})
    ax.set_title(f"Accuracy Heatmap — {eval_type.upper()}",
                 fontweight="bold")
    ax.set_xlabel("Target")
    ax.set_ylabel("Model")

plt.tight_layout()
plt.savefig("outputs/results/baseline_heatmap.png", bbox_inches="tight")
plt.show()

In [ ]:
# Rerun LOSO for best model (RF) — store per-subject detail
BEST_MODEL = "rf"
per_subject_records = []

for target in TARGETS:
    for test_sub in range(1, N_SUBJECTS + 1):
        train_mask = subjects != test_sub
        test_mask  = subjects == test_sub

        X_tr, y_tr = X[train_mask], y[target][train_mask]
        X_te, y_te = X[test_mask],  y[target][test_mask]

        if len(np.unique(y_te)) < 2:
            continue

        model = build_baseline(BEST_MODEL)
        model.fit(X_tr, y_tr)
        m = evaluate(model, X_te, y_te, split="loso")

        per_subject_records.append({
            "subject" : test_sub,
            "target"  : target,
            "accuracy": m["accuracy"],
            "f1"      : m["f1"],
            "roc_auc" : m["roc_auc"],
        })

df_per_subject = pd.DataFrame(per_subject_records)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ["#4C72B0", "#DD8452", "#55A868"]

for ax, target, color in zip(axes, TARGETS, colors):
    sub_df = df_per_subject[df_per_subject["target"] == target]
    ax.bar(sub_df["subject"], sub_df["accuracy"],
           color=color, edgecolor="white", alpha=0.85)
    ax.axhline(sub_df["accuracy"].mean(), color="red",
               linestyle="--", linewidth=1.5,
               label=f"Mean: {sub_df['accuracy'].mean():.3f}")
    ax.axhline(0.5, color="black", linestyle=":", linewidth=1,
               alpha=0.5, label="Chance")
    ax.set_title(f"LOSO Accuracy per Subject — {target.capitalize()}",
                 fontweight="bold")
    ax.set_xlabel("Subject ID")
    ax.set_ylabel("Accuracy")
    ax.set_xticks(range(1, N_SUBJECTS + 1))
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8)

plt.suptitle(f"Per-Subject LOSO Performance ({BEST_MODEL.upper()})",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/results/loso_per_subject.png", bbox_inches="tight")
plt.show()

In [ ]:
os.makedirs("outputs/models", exist_ok=True)

for target in TARGETS:
    # Train on ALL data with best model
    model = build_baseline(BEST_MODEL)
    model.fit(X, y[target])
    save_model(model, f"outputs/models/baseline_{BEST_MODEL}_{target}.pkl")

print("✅ Best baseline models saved for all three targets")

In [ ]:
results_summary = {
    "within": df_within.to_dict(orient="records"),
    "loso"  : df_loso.to_dict(orient="records"),
}
with open("outputs/results/baseline_results_full.json", "w") as f:
    json.dump(results_summary, f, indent=2)

if IN_COLAB:
    !git config --global user.email "you@example.com"
    !git config --global user.name "Your Name"
    !git add outputs/results/ outputs/models/ notebooks/
    !git commit -m "feat: baseline ML results — within + LOSO evaluation"
    !git push
    print("✅ Pushed to GitHub")